# 🥋 Poser — Dual-Stem Feature Fusion Network (5-Fold CV)
**Architecture:** Keras Functional API · Two Bi-LSTM Stems · Multi-Head Attention · Anti-Leakage 5-Fold Practitioner-Disjoint CV
**Classes:** GedanBarai · Gyakudzuki · MaeGeri

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ── Aesthetics ────────────────────────────────────────────────────────────────
plt.style.use('dark_background')
BG_COLOR = '#0D1117'
PALETTE  = {'GedanBarai': '#00C9FF', 'Gyakudzuki': '#FF6B6B', 'MaeGeri': '#A8FF78'}

## § 1 — Feature Column Definitions & Data Loader

In [ ]:
CSV_PATH = r"D:\CS-Senior26'\GR\martial-arts-trainer\backend\app\data\karate_normalized_base.csv"
df = pd.read_csv(CSV_PATH)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['folder_label'])
CLASS_NAMES = le.classes_.tolist()
N_CLASSES = len(CLASS_NAMES)

# Upper Body: landmarks 11-22 (x,y,z,v) + angle_08 to angle_13
UPPER_COORDS = [f'{c}{i}' for i in range(11, 23) for c in ['x', 'y', 'z', 'v']]
UPPER_ANGLES = [f'angle_{i:02d}' for i in range(8, 14)]
UPPER_FEATS  = UPPER_COORDS + UPPER_ANGLES   # 54

# Lower Body: landmarks 23-32 (x,y,z,v) + angle_00 to angle_07
LOWER_COORDS = [f'{c}{i}' for i in range(23, 33) for c in ['x', 'y', 'z', 'v']]
LOWER_ANGLES = [f'angle_{i:02d}' for i in range(0, 8)]
LOWER_FEATS  = LOWER_COORDS + LOWER_ANGLES   # 48

SEQ_LEN = 30

def extract_practitioner_and_view(clip_name):
    parts = clip_name.split('_')
    if len(parts) >= 4:
        return parts[2], parts[1] # practitioner_id, view
    return "Unknown", "Unknown"

clip_meta = []
for clip_id, group in df.groupby('clip_id'):
    practitioner, view = extract_practitioner_and_view(clip_id)
    clip_meta.append({
        'clip_id': clip_id,
        'label_enc': group['label_enc'].iloc[0],
        'practitioner_id': practitioner
    })
meta_df = pd.DataFrame(clip_meta)

def build_sequences(clips_list):
    sub_df = df[df['clip_id'].isin(clips_list)]
    clips = sub_df['clip_id'].unique()
    
    X_up = np.zeros((len(clips), SEQ_LEN, len(UPPER_FEATS)), dtype=np.float32)
    X_lo = np.zeros((len(clips), SEQ_LEN, len(LOWER_FEATS)), dtype=np.float32)
    y = np.zeros(len(clips), dtype=np.int32)
    
    for i, clip in enumerate(clips):
        grp = sub_df[sub_df['clip_id'] == clip].sort_values('frame_idx')
        X_up[i, :len(grp)] = grp[UPPER_FEATS].values[:SEQ_LEN]
        X_lo[i, :len(grp)] = grp[LOWER_FEATS].values[:SEQ_LEN]
        y[i] = grp['label_enc'].iloc[0]
        
    return X_up, X_lo, to_categorical(y, N_CLASSES), y


## § 2 — Dual-Stem Fusion Architecture Builder

In [ ]:
def build_dual_stem_model(n_upper=54, n_lower=48, seq_len=30, n_classes=3):
    inp_upper = keras.Input(shape=(seq_len, n_upper), name='upper_body')
    x_up = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inp_upper)
    x_up = layers.Dropout(0.4)(x_up)
    x_up = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x_up)

    inp_lower = keras.Input(shape=(seq_len, n_lower), name='lower_body')
    x_lo = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inp_lower)
    x_lo = layers.Dropout(0.4)(x_lo)
    x_lo = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x_lo)

    fused = layers.Concatenate(axis=-1)([x_up, x_lo])
    
    attn_out = layers.MultiHeadAttention(num_heads=4, key_dim=fused.shape[-1] // 4)(fused, fused)
    attn_out = layers.LayerNormalization()(attn_out + fused)
    attn_out = layers.Dropout(0.3)(attn_out)

    pooled = layers.GlobalAveragePooling1D()(attn_out)
    pooled = layers.Dense(64, activation='relu')(pooled)
    pooled = layers.Dropout(0.3)(pooled)
    output = layers.Dense(n_classes, activation='softmax')(pooled)

    return Model(inputs=[inp_upper, inp_lower], outputs=output)


## § 3 — 5-Fold Practitioner-Disjoint Cross-Validation
This perfectly separates practitioners so there is no data leakage across folds.

In [ ]:
gkf = GroupKFold(n_splits=5)

X_meta = meta_df['clip_id'].values
y_meta = meta_df['label_enc'].values
groups = meta_df['practitioner_id'].values

fold_accuracies = []
all_y_true = []
all_y_pred = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_meta, y_meta, groups), 1):
    train_clips = X_meta[train_idx]
    test_clips  = X_meta[test_idx]
    
    train_practitioners = set(groups[train_idx])
    test_practitioners  = set(groups[test_idx])
    
    print(f"\n{'='*60}")
    print(f"🥊 FOLD {fold}")
    print(f"Train Practitioners ({len(train_practitioners)}): {train_practitioners}")
    print(f"Test Practitioners  ({len(test_practitioners)}): {test_practitioners}")
    print(f"Leakage Overlap: {train_practitioners.intersection(test_practitioners)}")
    
    # Build tensors for this fold
    X_up_tr, X_lo_tr, y_tr_cat, _ = build_sequences(train_clips)
    X_up_te, X_lo_te, y_te_cat, y_te_true = build_sequences(test_clips)
    
    # Initialize fresh model per fold
    model = build_dual_stem_model()
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    
    es = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
    
    # Train
    print("Training model...")
    history = model.fit(
        [X_up_tr, X_lo_tr], y_tr_cat, 
        validation_data=([X_up_te, X_lo_te], y_te_cat), 
        epochs=100, batch_size=32, callbacks=[es], verbose=0
    )
    
    # Evaluate
    loss, acc = model.evaluate([X_up_te, X_lo_te], y_te_cat, verbose=0)
    print(f"⭐ Fold {fold} Test Accuracy: {acc * 100:.2f}%")
    fold_accuracies.append(acc)
    
    # Store predictions for the combined confusion matrix
    y_pred_probs = model.predict([X_up_te, X_lo_te], verbose=0)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    
    all_y_true.extend(y_te_true)
    all_y_pred.extend(y_pred_classes)

mean_acc = np.mean(fold_accuracies) * 100
std_acc = np.std(fold_accuracies) * 100
print(f"\n{'='*60}")
print(f"🏆 FINAL 5-FOLD CV RESULT: {mean_acc:.2f}% ± {std_acc:.2f}%")


## § 4 — Combined Confusion Matrix
This matrix aggregates the predictions from the test sets across all 5 folds.

In [ ]:
cm = confusion_matrix(all_y_true, all_y_pred)
plt.figure(figsize=(8, 6), facecolor=BG_COLOR)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', 
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cbar_kws={'shrink': 0.8}
)
plt.title('Aggregate Confusion Matrix (All 5 Folds)', color='white')
plt.ylabel('True Class', color='white')
plt.xlabel('Predicted Class', color='white')
plt.tick_params(colors='white')
plt.show()

print("\nClassification Report (Aggregate):\n")
print(classification_report(all_y_true, all_y_pred, target_names=CLASS_NAMES))
